# Adagrad — Adaptive Subgradient Methods for Online Learning and Stochastic Optimization (2011)

_Papers · Foundations & Optimization_

**Give every parameter its own learning rate that shrinks as its gradients accumulate — so rarely-seen, informative features still get big steps.**

---

This notebook is a practice scaffold for the **Adagrad — Adaptive Subgradient Methods for Online Learning and Stochastic Optimization (2011)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import torch

torch.manual_seed(0)

class MyAdagrad:
    """Diagonal Adagrad from scratch — eq. (1)/(5) of Duchi, Hazan & Singer (2011).
       Update: x -= lr * g / (sqrt(sum of past g^2) + eps)."""
    def __init__(self, params, lr=1e-2, eps=1e-10):
        self.params = list(params)
        self.lr, self.eps = lr, eps
        self.G = [torch.zeros_like(p) for p in self.params]   # accumulator (init 0)

    @torch.no_grad()
    def step(self):
        for i, p in enumerate(self.params):
            g = p.grad
            self.G[i] = self.G[i] + g*g                       # G_t = G_{t-1} + g^2 (LIFETIME sum)
            p -= self.lr * g / (self.G[i].sqrt() + self.eps)  # eps OUTSIDE the sqrt (= paper's delta)

    def zero_grad(self):
        for p in self.params:
            if p.grad is not None:
                p.grad.zero_()

# ---- THE ORACLE: MyAdagrad must equal torch.optim.Adagrad over several steps ----
w_mine = torch.randn(5, 3, requires_grad=True)
w_ref  = w_mine.detach().clone().requires_grad_(True)        # identical start
opt_mine = MyAdagrad([w_mine], lr=1e-2)                       # lr=1e-2, eps=1e-10
opt_ref  = torch.optim.Adagrad([w_ref], lr=1e-2)             # same defaults: eps=1e-10, init_acc=0

X = torch.randn(20, 5); target = torch.randn(20, 3)
for _ in range(6):
    opt_mine.zero_grad()
    (((X @ w_mine) - target)**2).mean().backward(); opt_mine.step()
    opt_ref.zero_grad()
    (((X @ w_ref) - target)**2).mean().backward(); opt_ref.step()

print("allclose vs torch.optim.Adagrad after 6 steps:",
      torch.allclose(w_mine, w_ref, atol=1e-7))             # expect True
print("max abs diff:", (w_mine - w_ref).abs().max().item())  # ~0

# ---- per-coordinate adaptation: rare feature keeps a BIGGER effective rate ----
# coord A active every step (g=1); coord B active only every 10th step
eta, eps = 0.1, 1e-10
G = torch.zeros(2)
for t in range(1, 31):
    g = torch.tensor([1.0, 1.0 if t % 10 == 0 else 0.0])
    G += g*g
eff = eta / (G.sqrt() + eps)
print("accum G  (frequent, rare):", G.tolist())             # [30.0, 3.0]
print("eff lr   (frequent, rare):", [round(x,4) for x in eff.tolist()])  # [0.0183, 0.0577]

# ---- recompute the two-step worked example: x0=0, g=2, eta=0.1 ----
x = torch.zeros(1)
o = MyAdagrad([x], lr=0.1, eps=0.0)
for t in (1, 2):
    x.grad = torch.tensor([2.0])
    o.step()
    print(f"t={t}: G={o.G[0].item():.4f}  x={x.item():.6f}")  # t1: G=4, x=-0.1 ; t2: G=8, x=-0.170711

## Visualize the data & results

_Minimize the same small, badly-conditioned least-squares loss from the same start with plain SGD vs Adagrad (each at a sensible learning rate) — does Adagrad's per-coordinate step reach a much lower loss in the same number of steps?_

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

# tiny ill-conditioned least-squares: features scaled unevenly -> one SGD lr struggles
N, D = 200, 6
scales = np.array([4.0, 3.0, 2.0, 1.0, 0.5, 0.25])
X = rng.normal(0, 1, (N, D)) * scales
w_true = rng.normal(0, 1, D)
y = X @ w_true + rng.normal(0, 0.01, N)

def loss_grad(w):
    r = X @ w - y
    return 0.5*np.mean(r**2), (X.T @ r)/N

def run_sgd(lr, steps=60):
    w = np.zeros(D); out = []
    for _ in range(steps):
        L, g = loss_grad(w); out.append(L); w -= lr*g
    return out

def run_adagrad(lr, steps=60, eps=1e-10):
    w = np.zeros(D); G = np.zeros(D); out = []
    for _ in range(steps):
        L, g = loss_grad(w); out.append(L)
        G += g*g                            # LIFETIME sum of squared gradients
        w -= lr*g/(np.sqrt(G)+eps)          # per-coordinate update
    return out

sgd  = run_sgd(lr=0.04)
ada  = run_adagrad(lr=0.5)
print("SGD     final loss:", round(sgd[-1], 4))    # ~0.1131
print("Adagrad final loss:", round(ada[-1], 4))    # ~0.0001

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** Take two coordinates over 30 steps. Coordinate A (frequent) has gradient 1 on every step; coordinate B (rare) has gradient 1 only on every 10th step (0 otherwise). With $\eta=0.1$, what is each one's effective learning rate after 30 steps, and what does that show?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Accumulate A: $G_A=\sum 1^2$ over all 30 steps $=30$. — _A is active every step._
- Accumulate B: $G_B=1^2\cdot 3=3$ (active on steps 10, 20, 30). — _B is active only 3 times._
- Effective rate A: $0.1/\sqrt{30}\approx0.0183$. — _Big pile $\Rightarrow$ small rate._
- Effective rate B: $0.1/\sqrt{3}\approx0.0577$. — _Small pile $\Rightarrow$ big rate._

**Answer:** The rare coordinate B keeps an effective rate about $3\times$ larger (0.0577 vs 0.0183). This is exactly the paper's claim: "frequently occurring features [get] very low learning rates and infrequent features high learning rates." Each of B's few updates "takes notice." The CODE cell prints these same two numbers.

</details>

**Problem 2.** Why does Adagrad typically beat plain SGD on a badly-conditioned problem (some directions much steeper than others)?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- On steep directions, gradients are large, so $\sqrt{G}$ is large and the step there shrinks. — _Prevents overshoot/oscillation in steep directions._
- On flat directions, gradients are small, so $\sqrt{G}$ is small and the step there stays large. — _Keeps making progress where SGD would crawl._
- SGD must use one global rate that is safe for the steepest direction, so it crawls in the flat ones. — _A single rate is a compromise._

**Answer:** Adagrad normalizes each coordinate's step by that coordinate's accumulated gradient size, so it can take large steps in flat directions and small steps in steep ones at the same time. Plain SGD, with one rate, must keep that rate small enough for the steepest direction and therefore crawls in the flat ones. In our CODEVIZ run Adagrad reaches loss ~0.0001 vs SGD's ~0.11 in 60 steps.

</details>

**Problem 3.** Ablation: in the CODE, replace the lifetime sum $G_t=G_{t-1}+g_t^2$ with a moving average $G_t=0.9\,G_{t-1}+0.1\,g_t^2$, then rerun the allclose against torch.optim.Adagrad. What happens, and what have you built?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Swap the accumulate line from a sum to a decayed moving average. — _That is no longer Adagrad's rule._
- Run the same 6 steps and call $\texttt{torch.allclose}$. — _PyTorch's Adagrad keeps the full sum._
- Note that the denominator $\sqrt{G_t}$ no longer grows without bound. — _The moving average forgets old gradients._

**Answer:** The allclose returns False: a moving average gives a different denominator than the lifetime sum, so the steps diverge from $\texttt{torch.optim.Adagrad}$. What you have built is essentially RMSProp &mdash; the moving-average fix that stops the learning rate from decaying to zero. This confirms two things: (1) Adagrad is specifically the cumulative sum, and (2) the cumulative sum is exactly why plain Adagrad stalls on long runs, which RMSProp was invented to fix.

</details>